# KG1 V70.5 — Max-Length Fix + Metric-Corrected Eval

**Date**: 2026-04-21

**Changes vs V70** (single variable + metric-aware eval):
- `max_length=8192` (was 4096; Tong recipe matches, reduces ~15% CoT truncation)
- Adds `enable_thinking=True` to chat template (was missing, bug)
- Uses CORRECTED metric for local eval (matches Kaggle official)

**Expected delta**: +0.015 (IC [+0.005, +0.025]) — Agent 6 audit

**Floor protection**: 0.840 absolute. Abort if smoke fails.

**Runtime**: ~4h A100 80GB / ~$5

## 0. Pre-flight checks (Agent V7 dossier)

- [ ] Colab Pro A100 (40GB VRAM minimum)
- [ ] HF_TOKEN in secrets
- [ ] KAGGLE_USERNAME + KAGGLE_KEY for eval data
- [ ] Baseline V70 adapter available for comparison

In [ ]:
# Cell 1 — Setup environment
!pip install -q "transformers>=4.55,<5.0" "peft>=0.13" "trl>=0.25" "bitsandbytes>=0.44" "accelerate>=1.0" "datasets>=3.0"
!pip install -q causal-conv1d mamba-ssm --no-build-isolation 2>/dev/null || echo "mamba-ssm skipped (fallback to slow path)"

import os, sys, subprocess, json, time
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# HF auth
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN') or userdata.get('HF_KEY')
assert HF_TOKEN, 'HF_TOKEN required'
os.environ['HF_TOKEN'] = HF_TOKEN

# Clone KG1 code (pull from GitHub/fork or use rsync)
REPO_PATH = '/content/kg1'
if not Path(REPO_PATH).exists():
    # User should rsync their repo or clone from their private fork
    print('Clone/rsync your KG1 repo to /content/kg1')
    print('Example: !git clone https://github.com/<user>/KG1-NVIDIA /content/kg1')

sys.path.insert(0, REPO_PATH)
print('Setup OK')

In [ ]:
# Cell 2 — Training config (V70.5 = V70 + max_length fix)
CFG = {
    'model': 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16',
    'adapter_out': 'felipesp1983/kg1-nemotron-lora-v70-5-maxlen-fix',
    
    # LoRA (V70 proven)
    'lora_rank': 32,
    'lora_alpha': 32,
    'lora_dropout': 0.0,
    'target_modules': 'all-linear',
    'bias': 'none',
    
    # Training (V70 proven, ONLY max_length changes)
    'epochs': 1,
    'lr': 2e-4,
    'batch_size': 1,
    'grad_accum': 64,  # effective batch 64
    'max_length': 8192,  # WAS 4096 — V70.5 FIX
    'seed': 42,
    'optim': 'paged_adamw_8bit',
    'gradient_checkpointing': True,
    'attn_implementation': 'eager',  # Mamba hybrid requires eager
    
    # Data
    'train_data': 'felipesp1983/kg1-nemotron-training/data/sft_v70_huikang_full.jsonl',
}

print(json.dumps(CFG, indent=2))

In [ ]:
# Cell 3 — Train with V70.5 config
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch

# Load model
model = AutoModelForCausalLM.from_pretrained(
    CFG['model'],
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation=CFG['attn_implementation'],
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

tokenizer = AutoTokenizer.from_pretrained(CFG['model'], trust_remote_code=True)

# LoRA
lora_cfg = LoraConfig(
    r=CFG['lora_rank'],
    lora_alpha=CFG['lora_alpha'],
    target_modules=CFG['target_modules'],
    lora_dropout=CFG['lora_dropout'],
    bias=CFG['bias'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# Load data
dataset = load_dataset('json', data_files=CFG['train_data'], split='train')
print(f'Training rows: {len(dataset)}')

# SFT config
sft_cfg = SFTConfig(
    output_dir='/content/v70_5_output',
    num_train_epochs=CFG['epochs'],
    per_device_train_batch_size=CFG['batch_size'],
    gradient_accumulation_steps=CFG['grad_accum'],
    learning_rate=CFG['lr'],
    lr_scheduler_type='linear',
    warmup_ratio=0.03,
    adam_beta1=0.9,
    adam_beta2=0.95,
    weight_decay=0.0,
    optim=CFG['optim'],
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    max_length=CFG['max_length'],  # V70.5 FIX
    completion_only_loss=True,
    packing=False,
    seed=CFG['seed'],
    bf16=True,
    logging_steps=1,
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=sft_cfg,
    train_dataset=dataset,
    processing_class=tokenizer,
)

# SMOKE TEST first (2 steps only)
import copy
smoke_cfg = copy.deepcopy(sft_cfg)
smoke_cfg.max_steps = 2
print('=== SMOKE TEST (2 steps) ===')
trainer.args.max_steps = 2
trainer.train()
# Check loss sanity
history = trainer.state.log_history
losses = [h['loss'] for h in history if 'loss' in h]
print(f'Smoke losses: {losses}')
assert max(losses) < 50 and min(losses) > 0, 'ABORT: loss diverges in smoke'
print('SMOKE OK — proceeding with full train')

In [ ]:
# Cell 4 — Full training (1 epoch)
trainer.args.max_steps = -1  # disable smoke limit
trainer.train()

# Save adapter
trainer.save_model('/content/v70_5_final')
print('Training complete. Adapter saved to /content/v70_5_final')

In [ ]:
# Cell 5 — Local eval with CORRECTED metric (D1 reverse engineering)
# This uses scripts/local_score.py with the fixes applied 2026-04-21

import subprocess
result = subprocess.run([
    'python', f'{REPO_PATH}/scripts/local_score.py',
    '--adapter', '/content/v70_5_final',
    '--n-samples', '600',
    '--output-csv', '/content/v70_5_eval.csv',
], capture_output=True, text=True)
print(result.stdout[-3000:])
print('STDERR:', result.stderr[-1000:])

# Load eval results
import pandas as pd
eval_df = pd.read_csv('/content/v70_5_eval.csv')
print(f'\nOverall accuracy: {eval_df["correct"].mean():.4f}')
print('\nPer category:')
if 'category' in eval_df.columns:
    print(eval_df.groupby('category')['correct'].mean())

# DECISION GATE (99% rule)
overall = eval_df['correct'].mean()
V70_FLOOR = 0.82  # conservative; real V70 local was 0.84 with OLD metric
if overall >= V70_FLOOR:
    print(f'\nGATE: GO — local score {overall:.4f} >= {V70_FLOOR}')
    print('Proceed to upload adapter to HF and gate+submit to Kaggle')
else:
    print(f'\nGATE: NO-GO — local score {overall:.4f} < {V70_FLOOR}')
    print('ABORT: something regressed vs V70 baseline')

In [ ]:
# Cell 6 — Upload adapter to HF (only if gate passes)
from huggingface_hub import HfApi

overall = eval_df['correct'].mean()
if overall >= 0.82:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(CFG['adapter_out'], repo_type='model', private=True, exist_ok=True)
    api.upload_folder(
        folder_path='/content/v70_5_final',
        repo_id=CFG['adapter_out'],
        repo_type='model',
        commit_message=f'V70.5 max_length=8192 fix, local eval {overall:.4f}',
    )
    print(f'Uploaded to: https://huggingface.co/{CFG["adapter_out"]}')
else:
    print('SKIPPED upload due to gate failure')

## Next Steps

If V70.5 gate passes AND Kaggle LB > 0.84:
- Proceed to V71.1 (max-min-logprob loss)
- Then V71.2 (bit pairs CoT)
- Then V71.3 (cryptarithm 47-combo CoT)
- Then V71 (PEFT 3 fixes: rsLoRA + PiSSA + modules_to_save)

If V70.5 gate FAILS:
- Max-length 8192 may not help our recipe
- Investigate VRAM issue or data quality
- Rollback to V70

See `ROADMAP_V71_TOP1_ALTA_CONFIANCA_v4.md` for full pipeline.